# Reproducing Iram, Garcia et al. (2024) from dandiset 001350

**Paper.** Iram T, Garcia MA, Amand J, et al. (2024). SRF transcriptionally regulates the
oligodendrocyte cytoskeleton during CNS myelination. *PNAS* 121(12), e2307250121.
[doi:10.1073/pnas.2307250121](https://doi.org/10.1073/pnas.2307250121). The claims file
(`claims/001350/10.1073-pnas.2307250121.yaml`) was written from the version of record in Europe PMC
(PMC10962977) with the SI Appendix; this attempt read the same text and SI Appendix.

**Dandiset.** [DANDI:001350](https://dandiarchive.org/dandiset/001350/0.250511.1527), version
0.250511.1527, "TEM Images of Corpus Callosum in Flox/SRF-cKO Mice". The paper cites no DANDI
version (its data statement names only NCBI GEO GSE241561), and this is the only published version,
so it is pinned. The draft lists the same 134 assets and the same total size; it is not used.

**What the dandiset holds.** A BIDS microscopy dataset (no NWB files): 8-bit PNG images of adult
corpus callosum at 8000x from 5 SRF-Flox ("WT") and 5 SRF-cKO ("KO") mice, and, under
`derivatives/labels`, masks of myelinated axons with their myelin, unmyelinated axons,
oligodendrocyte nuclei and processes for some images. The masks were made for a later segmentation
project (AxonDeepSeg pseudo-labels corrected by hand), not by the paper. These are the data of the
paper's Figure 2A to 2E and Fig. S3A to S3D. The optic nerve TEM, immunostaining, RNAscope,
culture imaging and sequencing data are not in it; the sequencing data are in GEO and are
tested in `geo_sequencing_claims.ipynb`.

**Method.** The paper scored and measured axons by hand in Fiji. Here the dataset's masks stand in
for that scoring (reading O1 of the claims file), and every other reading the claims file names
(O2 to O7) is computed. Loading and measures are in `srf_tem.py`, whose docstring says where each
parameter comes from.

| Claim | Data | Status |
|---|---|---|
| fig2c-cc-percent-myelinated | dandiset | analysed below; blocked: data missing (masks for one SRF-Flox mouse) |
| fig2d-cc-myelinated-density | dandiset | analysed below; blocked: data missing (masks for one SRF-Flox mouse) |
| fig2e-cc-g-ratio | dandiset | analysed below; blocked: data missing (masks for one SRF-Flox mouse) |
| figs3c-cc-myelin-thickness-unchanged | dandiset | analysed below; blocked: data missing (masks for one SRF-Flox mouse) |
| figs3d-cc-larger-myelinated-diameter | dandiset | analysed below; blocked: data missing (masks for one SRF-Flox mouse) |
| figs2b, figs2e, figs2cf, figs2ij, figs2kl (optic nerve TEM) | dandiset | blocked: data missing |
| fig2gk, figs3fg, fig1d, fig1ef, figs1, fig4eg (immunostaining, RNAscope, RT-PCR) | dandiset | blocked: data missing |
| fig5e, fig5d, culture-mbp-unchanged (culture imaging) | dandiset | blocked: data missing |
| fig3a, fig3bc, fig3d, no-myelin-genes, fig4ab, fig4cd, figs4ef | elsewhere (GEO GSE241561) | tested in `geo_sequencing_claims.ipynb` |
| figs5ab, fig6b, fig6cd, fig6eg | elsewhere (GEO GSE241561) | blocked: not runnable (see `geo_sequencing_claims.ipynb`) |

In [ ]:
import time
import json
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy import stats

import srf_tem

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
meta = srf_tem.version_metadata()
assets = srf_tem.assets()
imgs = srf_tem.images()
participants = srf_tem.read_tsv("participants.tsv")
samples = srf_tem.read_tsv("samples.tsv")
genotype = dict(zip(participants.participant_id, participants.phenotype))
imgs["genotype"] = imgs.subject.map(genotype)
print(f"dandiset {srf_tem.DANDISET_ID} version {srf_tem.DANDISET_VERSION}: {meta['name']}")
print(f"{len(assets)} assets, {assets['size'].sum() / 1e9:.2f} GB")

## Data check

Each check of the metadata against the files is stated under its output.

In [ ]:
summary = meta["assetsSummary"]
print("Archive summary: subjects", summary.get("numberOfSubjects"), "| samples", summary.get("numberOfSamples"),
      "| files", summary.get("numberOfFiles"), "| species", summary.get("species"),
      "| standard", [d["name"] for d in summary.get("dataStandard", [])])
print("\nparticipants.tsv:")
print(participants.to_string(index=False))
print("\nGenotypes:", participants.phenotype.value_counts().to_dict())
print("Species values:", participants.species.unique().tolist())
print("samples.tsv rows:", len(samples), "| raw images on the archive:", len(imgs))
print("samples.tsv rows without an image:",
      len(set(zip(samples.participant_id, samples.sample_id)) - set(zip(imgs.subject, imgs["sample"]))),
      "| images without a samples.tsv row:",
      len(set(zip(imgs.subject, imgs["sample"])) - set(zip(samples.participant_id, samples.sample_id))))

The archive summary counts 10 subjects, as the files hold, but 16 samples, where `samples.tsv` has
86 rows, one per image (the two small regions of sub-373C included), and it lists no species. `participants.tsv` gives the species as "mus muculus"
(a misspelling of *Mus musculus*). The genotypes are 5 WT and 5 KO, as the description and the paper
(n = 5 per genotype) say; the labels WT and KO stand for SRF-Flox and SRF-cKO.

In [ ]:
def png_size(url):
    import requests, struct
    head = requests.get(url, headers={"Range": "bytes=0-31"}, timeout=60).content
    w, h = struct.unpack(">II", head[16:24])
    return h, w

with ThreadPoolExecutor(16) as ex:
    shapes = list(ex.map(png_size, assets.set_index("path").loc[imgs.path, "url"]))
imgs["rows"], imgs["cols"] = zip(*shapes)
px = {s: srf_tem.pixel_size_um(s) for s in imgs.subject.unique()}
print("Pixel size by subject (um):", px)
imgs["area_um2"] = imgs.rows * imgs.cols * imgs.subject.map(px) ** 2
imgs["labelled"] = imgs.axonmyelin.notna() & imgs.uaxon.notna()
per_subject = imgs.groupby("subject").agg(
    genotype=("genotype", "first"), images=("path", "size"),
    image_rows=("rows", lambda r: f"{r.min()}-{r.max()}"), image_cols=("cols", lambda c: f"{c.min()}-{c.max()}"),
    imaged_um2=("area_um2", "sum"), labelled_images=("labelled", "sum"),
    labelled_um2=("area_um2", lambda a: a[imgs.loc[a.index, "labelled"]].sum()))
print(per_subject.round(1).to_string())
print("\nMasks by kind:", {k: int(imgs[k].notna().sum()) for k in ["axonmyelin", "uaxon", "nuclei", "process"]})
print("Labelled images by genotype:", imgs[imgs.labelled].groupby("genotype").subject.agg(lambda s: sorted(set(s))).to_dict())

Every subject's sidecar gives the same pixel size, 0.00493 um. Most images are 4675 x 4675 pixels
(531 um2 at that scale); sub-366A's sixteen images are smaller (1482 to 2385 rows, 1648 to 2613
columns) and sub-367A to sub-369B's are larger (3412 to 5058 rows, 3712 to 5712 columns). Every mouse
has more than 1000 um2 of images, the paper's minimum, so the images could support the paper's
design. The masks do not: both kinds of axon mask exist for 8 full images and 2 small regions, from
one SRF-Flox mouse (sub-370) and four SRF-cKO mice. The scale cannot be checked against a scale bar,
as the images carry none; the measured axon sizes below are consistent with it.

In [ ]:
def digest(asset_id):
    return srf_tem.asset_metadata(asset_id)["digest"]["dandi:sha2-256"]

pngs = assets[assets.path.str.endswith(".png")]
with ThreadPoolExecutor(16) as ex:
    digests = dict(zip(pngs.path, ex.map(digest, pngs.asset_id)))
dup = pd.Series(digests).duplicated(keep=False)
print(f"PNG files: {len(digests)}; files sharing a SHA-256 with another: {int(dup.sum())}")

# The two small regions of sub-373C against the README's coordinates (x, y of two corners).
full = srf_tem.read_png("sub-373C/micr/sub-373C_sample-0001_TEM.png")
for chunk, (x0, y0, x1, y1) in {"01": (800, 1500, 1400, 2100), "02": (2400, 1300, 2900, 1800)}.items():
    roi = srf_tem.read_png(f"sub-373C/micr/sub-373C_sample-0001_acq-roi_chunk-{chunk}_TEM.png")
    print(f"chunk-{chunk} {roi.shape}: equal to sample-0001 rows {y0}:{y1}, columns {x0}:{x1}:",
          np.array_equal(full[y0:y1, x0:x1], roi))

# Overlap between labelled images of the same mouse, by phase correlation, with a known overlap as control.
raw = {p: srf_tem.read_png(p) for p in imgs[imgs.labelled & (imgs.rows > 1000)].path}
ctrl = next(iter(raw.values()))
print(f"\ncontrol, two crops of one image overlapping by 3000 x 2000 px: peak {srf_tem.phase_peak(ctrl[:3000, :3000], ctrl[1500:4500, 1000:4000])[0]:.3f}")
rows = []
paths = list(raw)
for i, a in enumerate(paths):
    for b in paths[i + 1:]:
        peak, shift = srf_tem.phase_peak(raw[a], raw[b])
        rows.append({"a": a.split("/")[-1], "b": b.split("/")[-1], "same_mouse": a.split("/")[0] == b.split("/")[0],
                     "peak": round(peak, 4), "shift": shift})
overlap = pd.DataFrame(rows)
print(overlap.sort_values("peak", ascending=False).head(8).to_string(index=False))
print("pairs:", len(overlap), "| largest peak:", overlap.peak.max())
del raw

No two files are identical. The two regions of sub-373C are exact crops of its sample-0001 at the
README's coordinates, read as (column, row); sample-0001 itself has no masks, so the regions are
counted once, as part of sub-373C. No pair of labelled full images shares content: the largest
phase-correlation peak between them is printed above, against the control's for a known overlap.

In [ ]:
measures = {}
t0 = time.time()
for r in imgs[imgs.labelled].itertuples():
    am, ua = srf_tem.read_png(r.axonmyelin), srf_tem.read_png(r.uaxon)
    assert am.shape == (r.rows, r.cols) == ua.shape
    m = srf_tem.measure_image(am, ua, px[r.subject])
    m["values_axonmyelin"] = sorted(np.unique(am).tolist())
    m["values_uaxon"] = sorted(np.unique(ua).tolist())
    measures[(r.subject, r.sample)] = m
print(f"measured {len(measures)} labelled images in {time.time() - t0:.0f} s")
check = pd.DataFrame([{
    "subject": s, "sample": smp, "genotype": genotype[s], "area_um2": round(m["area_um2"], 1),
    "axonmyelin_values": m["values_axonmyelin"], "uaxon_values": m["values_uaxon"],
    "myelinated": len(m["myelinated"]), "unmyelinated": len(m["unmyelinated"]),
    "axon_specks": m["axon_specks_dropped"], "uaxon_specks": m["uaxon_specks_dropped"],
    "axons_no_myelin": m["axons_without_myelin"], "mask_overlap_px": m["overlap_px"],
    "myelin_unassigned_pct": round(100 * m["myelin_unassigned_px"] / max(m["myelin_px"], 1), 3)}
    for (s, smp), m in measures.items()])
print(check.to_string(index=False))

The axon-and-myelin masks hold axon at 255 (254 in sub-373C chunk-01, which is stored as grey with
alpha) and myelin at 127, with a few pixels at 126; the unmyelinated-axon masks are binary. In the
two SRF-Flox images no pixel is in both kinds of axon mask; in the cKO images up to 51,098 pixels
(0.23% of an image) are. Objects under 20 pixels are dropped as specks (counts above). Myelin not
connected to any axon is at most 0.26% of an image's myelin, and every myelinated axon has some
myelin assigned.

### The masks on their images

The masks were made for a segmentation project, so whether they label every axon in an image is not
documented. The figure shows a 1000 x 1000 pixel crop (4.93 um a side) from the centre of one
SRF-Flox and one SRF-cKO image, the raw image above and the mask outlines below (blue: myelinated
axon, orange: outer edge of its myelin, magenta: unmyelinated axon). Grey levels are shown on their
full 0 to 255 range; nothing falls outside it. The counts printed are the objects that touch each crop.

In [ ]:
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.measure import label as cc_label
from skimage.segmentation import find_boundaries

overlay_images = [("sub-370", "sample-0006"), ("sub-375", "sample-0005")]
fig, axes = plt.subplots(2, 2, figsize=(11, 11))
for j, (s, smp) in enumerate(overlay_images):
    r = imgs[(imgs.subject == s) & (imgs["sample"] == smp)].iloc[0]
    raw = srf_tem.read_png(r.path)
    am, ua = srf_tem.read_png(r.axonmyelin), srf_tem.read_png(r.uaxon)
    r0, c0, w = raw.shape[0] // 2 - 500, raw.shape[1] // 2 - 500, 1000
    sl = (slice(r0, r0 + w), slice(c0, c0 + w))
    axon, myelin = srf_tem.split_axonmyelin(am[sl])
    unmy = ua[sl] > 0
    grey = raw[sl].astype(float) / 255
    rgb = np.repeat(grey[..., None], 3, -1)
    for mask, colour in [((axon | myelin), (0.95, 0.55, 0.1)), (axon, (0.15, 0.45, 0.95)), (unmy, (0.9, 0.1, 0.8))]:
        edge = ndimage.binary_dilation(find_boundaries(cc_label(mask, connectivity=1), mode="inner"))
        rgb[edge] = colour
    axes[0, j].imshow(grey, cmap="gray", vmin=0, vmax=1)
    axes[1, j].imshow(rgb)
    for ax in axes[:, j]:
        ax.set_xticks([]); ax.set_yticks([])
    axes[0, j].set_title(f"{s} ({genotype[s]}) {smp}, rows {r0}-{r0 + w}, cols {c0}-{c0 + w}")
    n_my = cc_label(axon, connectivity=1).max()
    n_un = cc_label(unmy, connectivity=1).max()
    area = (w * px[s]) ** 2
    print(f"{s} {genotype[s]} {smp}: crop {area:.1f} um2, grey {raw[sl].min()} to {raw[sl].max()}; "
          f"myelinated axons {n_my} ({n_my / area:.2f} per um2), unmyelinated axon objects {n_un} "
          f"({n_un / area:.2f} per um2)")
plt.tight_layout()
plt.show()

In both crops every myelinated fibre visible in the image has an axon and a myelin outline, and
nearly every small unmyelinated profile has an outline; the masks do not look incomplete. This is a
check by eye on two crops of 24 um2, not a measure of completeness over the 4264 um2 the masks cover.
The SRF-cKO crop includes part of a cell body (lower right), which holds no axons.

In [ ]:
axons = pd.concat([m["myelinated"].assign(subject=s, sample=smp, genotype=genotype[s])
                   for (s, smp), m in measures.items()], ignore_index=True)
unmy = pd.concat([m["unmyelinated"].assign(subject=s, sample=smp, genotype=genotype[s])
                  for (s, smp), m in measures.items()], ignore_index=True)
q = [0.05, 0.5, 0.95]
print("Myelinated axon diameter, equivalent circle (nm), 5th, 50th and 95th percentiles by genotype:")
print(axons.groupby("genotype").axon_d_eq_nm.quantile(q).unstack().round(0).to_string())
print("\nMyelin thickness, equivalent circle (nm):")
print(axons[axons.myelin_px > 0].groupby("genotype").thick_eq_nm.quantile(q).unstack().round(0).to_string())
print("\nUnmyelinated axon object diameter, equivalent circle (nm):")
print(unmy.groupby("genotype").d_eq_nm.quantile(q).unstack().round(0).to_string())
big = unmy.d_eq_nm > 600
print(f"\nUnmyelinated objects over 600 nm (possibly merged neighbours): {int(big.sum())} of {len(unmy)} ({100 * big.mean():.1f}%)")
print(f"Myelinated axon g-ratio (equivalent circle) range: {axons.g_eq.min():.3f} to {axons.g_eq.max():.3f}")
print("\nPer image: myelinated, unmyelinated, total objects")
print(check.assign(total=check.myelinated + check.unmyelinated)[["subject", "sample", "myelinated", "unmyelinated", "total"]].to_string(index=False))
print("\nPer mouse total objects:", check.assign(total=check.myelinated + check.unmyelinated).groupby("subject").total.sum().to_dict())

With the stated pixel size, myelinated axons in adult corpus callosum measure a few hundred
nanometres to about a micrometre, myelin about 100 nm, and unmyelinated axons about 0.2 um, sizes
consistent with adult mouse corpus callosum; this is consistent with the stated scale, not a proof of
it. The images have no time base, so the protocol's check of a shared clock does not apply.

## How the claims are tested

The claims file's criteria call for per-mouse values from 5 SRF-Flox and 5 SRF-cKO mice and an
unpaired two-tailed t test (Methods, "Data Analysis and Statistics": "Statistical significance was
determined between biological replicate ... by an unpaired, two-tailed t test"; Figure 2C to 2E
legends, "n = 5 animals from each genotype"). The paper's per-axon scoring is not deposited, and the
masks that stand in for it (reading O1) cover one SRF-Flox mouse (sub-370) and four SRF-cKO mice.
The per-mouse measurements the test needs are missing for four of the five SRF-Flox mice and one of
the five SRF-cKO mice, so each claim below is blocked as `data_missing`. A t test with one Flox mouse
has no estimate of the Flox spread: Student's test borrows the cKO variance, and Welch's test is
undefined. Each section still shows every value the masks allow, under every reading the claims file
names: the per-mouse values, the group means against the expected values, the Student t test of the
one Flox mouse against the cKO mice (not the paper's design), and, as a variant, tests over images
(which treat images as independent, unlike the paper).

The dandiset's description points to a segmentation model trained on these masks
(github.com/axondeepseg/model_seg_unmyelinated_tem), which could label the other 76 full images. It is not
used. The paper does not rely on it (the claims file says it is not the paper's code), so under the
protocol it cannot be brought in as an outside input; it was trained on the very images labelled
here, and its uncorrected output on the other mice would be a further step from the paper's hand
scoring than the hand-corrected masks are. It is an nnU-Net model meant for a GPU; this runner has
4 CPUs and none.

sub-373C has only two small regions (15 um2), far below the paper's minimum area; it is shown, and
each result is also given without it.

In [ ]:
def student_one_vs_many(x_cko, x_flox):
    "Student's pooled t test, which is defined when one group has a single value; Welch's is not."
    a, b = np.asarray(x_cko, float), np.asarray(x_flox, float)
    na, nb = len(a), len(b)
    ss = ((a - a.mean()) ** 2).sum() + ((b - b.mean()) ** 2).sum()
    df = na + nb - 2
    sp = np.sqrt(ss / df)
    t = (a.mean() - b.mean()) / (sp * np.sqrt(1 / na + 1 / nb))
    return t, df, 2 * stats.t.sf(abs(t), df)

def welch(x_cko, x_flox):
    if min(len(x_cko), len(x_flox)) < 2:
        return np.nan, np.nan
    r = stats.ttest_ind(x_cko, x_flox, equal_var=False)
    return r.statistic, r.pvalue

def group_table(values, name, expected_flox, expected_cko):
    "values: per-mouse Series indexed by subject. Group means, tests, with and without sub-373C."
    rows = []
    for label, keep in [("all labelled mice", values.index), ("without sub-373C", values.index.drop("sub-373C", errors="ignore"))]:
        v = values.loc[keep]
        flox = v[[genotype[s] == "WT" for s in v.index]]
        cko = v[[genotype[s] == "KO" for s in v.index]]
        t, df, p = student_one_vs_many(cko, flox)
        wt, wp = welch(cko, flox)
        rows.append({"mice": label, "n_flox": len(flox), "n_cko": len(cko),
                     "flox_mean": flox.mean(), "cko_mean": cko.mean(), "cko_minus_flox": cko.mean() - flox.mean(),
                     "cko_over_flox": cko.mean() / flox.mean(),
                     "flox_in_tol": abs(flox.mean() - expected_flox[0]) <= expected_flox[1],
                     "cko_in_tol": abs(cko.mean() - expected_cko[0]) <= expected_cko[1],
                     "student_t": t, "df": df, "student_p": p, "welch_p": wp})
    return pd.DataFrame(rows).set_index("mice")

def image_test(per_image, col):
    "Variant, not the paper's design: images as units."
    f = per_image[per_image.genotype == "WT"][col]
    c = per_image[per_image.genotype == "KO"][col]
    return {"n_images_flox": len(f), "n_images_cko": len(c), "flox_mean": f.mean(), "cko_mean": c.mean(),
            "student_p": stats.ttest_ind(c, f).pvalue, "welch_p": stats.ttest_ind(c, f, equal_var=False).pvalue}

## fig2c-cc-percent-myelinated

> In the adult corpus callosum SRF-cKO mice have a lower percentage of myelinated axons than
> SRF-Flox littermates (about 22% against about 37%, read from the figure), so the hypomyelination
> seen in optic nerve persists into adulthood in corpus callosum.

Expected: SRF-Flox 37 +/- 3.7%, SRF-cKO 22 +/- 2.2%; unpaired t test p < 0.05 under readings O1,
O2, O4 and O7; axons counted per mouse reported against the paper's minimum of 2,500.

In [ ]:
counts = {}
for keep_edge in [False, True]:
    rows = [{"subject": s, "sample": smp, "genotype": genotype[s], **srf_tem.per_image_counts(m, keep_edge)}
            for (s, smp), m in measures.items()]
    counts[keep_edge] = pd.DataFrame(rows)
pd_tables = {k: srf_tem.mouse_percent_and_density(v) for k, v in counts.items()}
for keep_edge, tab in pd_tables.items():
    print(f"O4 ({'b' if keep_edge else 'a'}): edge objects {'kept' if keep_edge else 'left out'}")
    print(tab.assign(genotype=[genotype[s] for s in tab.index]).round(3).to_string())
    print()

In [ ]:
fig2c = {}
for keep_edge, tab in pd_tables.items():
    for col in ["pct_pooled_O2a", "pct_image_mean_O2b"]:
        key = f"O4{'b' if keep_edge else 'a'} {col}"
        fig2c[key] = group_table(tab[col], col, (37, 3.7), (22, 2.2))
        print(key); print(fig2c[key].round(4).to_string()); print()
allc = pd.concat(fig2c)
print(f"{len(allc)} combinations: Flox {allc.flox_mean.min():.4g} to {allc.flox_mean.max():.4g}, cKO {allc.cko_mean.min():.4g} to "
      f"{allc.cko_mean.max():.4g}, cKO below Flox in {int((allc.cko_minus_flox < 0).sum())}, Student p {allc.student_p.min():.3g} to {allc.student_p.max():.3g}\n")
print("Axons counted per mouse (edges left out), against the paper's minimum of 2,500:")
print(pd_tables[False].axons_counted.to_dict())
print("\nVariant, images as units (not the paper's design):")
for keep_edge in [False, True]:
    c = counts[keep_edge].assign(pct=lambda d: 100 * d.n_myelinated / (d.n_myelinated + d.n_unmyelinated))
    print(f"O4{'b' if keep_edge else 'a'}", {k: round(v, 4) for k, v in image_test(c, "pct").items()})

print("\nVariant, unmyelinated axons counted with the pixels of myelinated fibres removed, so no pixel is both:")
for keep_edge in [False, True]:
    ex = pd.DataFrame([{"subject": s, "sample": smp, **srf_tem.per_image_counts(m, keep_edge, exclusive=True)}
                       for (s, smp), m in measures.items()])
    tab = srf_tem.mouse_percent_and_density(ex)
    print(f"O4{'b' if keep_edge else 'a'} percent myelinated (O2a):", tab.pct_pooled_O2a.round(2).to_dict(),
          "| change from the main count:", (tab.pct_pooled_O2a - pd_tables[keep_edge].pct_pooled_O2a).round(2).to_dict())

**Result: blocked (data missing: masks for one SRF-Flox mouse).** The claim's per-mouse test of 5
against 5 mice cannot be run. That mouse, sub-370, has 26.3% of its axons myelinated (readings O2a
and O4a; 26.3% to 28.2% over the four readings), below the expected 37 +/- 3.7%. The four cKO mice
average 14.3% (14.3% to 18.9%), below the expected 22 +/- 2.2%, and lower than the Flox mouse under
every reading, with Student p 0.13 to 0.20 for one Flox mouse against the cKO mice. With images as
units, Student p is 0.014 (O4a) and 0.036 (O4b), Welch p 0.12 and 0.19. sub-370 has 3,842 axons
counted, above the paper's 2,500. Counting unmyelinated axons without the pixels that are also in a
myelinated fibre changes no mouse's percent by more than 0.02 points. The direction of the paper's
effect is present in these masks; both genotypes are below the paper's values.

## fig2d-cc-myelinated-density

> SRF-cKO adult corpus callosum has fewer myelinated axons per area than SRF-Flox (about 1.4
> against about 1.8 per unit area, read from the figure).

Expected: SRF-Flox 1.8 +/- 0.18 and SRF-cKO 1.42 +/- 0.142 per um2 (reading O3), ratio 0.79 +/-
0.079; imaged area per mouse reported against 1,000 um2.

In [ ]:
fig2d = {}
for keep_edge, tab in pd_tables.items():
    for col in ["dens_pooled_O3a", "dens_image_mean_O3b"]:
        key = f"O4{'b' if keep_edge else 'a'} {col}"
        fig2d[key] = group_table(tab[col], col, (1.8, 0.18), (1.42, 0.142))
        g = fig2d[key]
        print(key); print(g.round(4).to_string())
        print("  ratio within 0.79 +/- 0.079:", (abs(g.cko_over_flox - 0.79) <= 0.079).to_dict()); print()
alld = pd.concat(fig2d)
print(f"{len(alld)} combinations: Flox {alld.flox_mean.min():.4g} to {alld.flox_mean.max():.4g}, cKO {alld.cko_mean.min():.4g} to "
      f"{alld.cko_mean.max():.4g}, ratio {alld.cko_over_flox.min():.3g} to {alld.cko_over_flox.max():.3g}, Student p {alld.student_p.min():.3g} to {alld.student_p.max():.3g}")
print("Labelled area per mouse (um2), against the paper's minimum of 1,000:", pd_tables[False].area_um2.round(1).to_dict())
print("Myelinated axons per 100 um2, per mouse (O3a, O4a):", (100 * pd_tables[False].dens_pooled_O3a).round(1).to_dict())
print("\nVariant, images as units (not the paper's design):")
for keep_edge in [False, True]:
    c = counts[keep_edge].assign(dens=lambda d: d.n_myelinated / d.area_um2)
    print(f"O4{'b' if keep_edge else 'a'}", {k: round(v, 4) for k, v in image_test(c, "dens").items()})

In [ ]:
# All axons per um2 in the masks, against what the paper's two panels imply together
# (myelinated per um2 divided by the fraction myelinated: Fig. 2D over Fig. 2C).
tot = pd_tables[False].assign(
    genotype=[genotype[s] for s in pd_tables[False].index],
    all_axons_per_um2=lambda t: t.axons_counted / t.area_um2,
    myelinated_per_um2=lambda t: t.dens_pooled_O3a)
print(tot[["genotype", "area_um2", "axons_counted", "all_axons_per_um2", "pct_pooled_O2a", "myelinated_per_um2"]].round(3).to_string())
print(f"\nImplied by the paper's values (O3 unit): SRF-Flox {1.8 / 0.37:.2f} and SRF-cKO {1.42 / 0.22:.2f} axons per um2 in all; "
      f"the paper's minimums (2,500 axons over 1,000 um2) imply at least 2.5")
for g in ["WT", "KO"]:
    t = tot[(tot.genotype == g) & (tot.index != "sub-373C")]
    print(f"Masks, {g} (without sub-373C): all axons {t.all_axons_per_um2.min():.2f} to {t.all_axons_per_um2.max():.2f} per um2, "
          f"myelinated {t.myelinated_per_um2.min():.2f} to {t.myelinated_per_um2.max():.2f} per um2")

**Result: blocked (data missing: masks for one SRF-Flox mouse).** sub-370 has 0.95 myelinated axons
per um2 (1.04 with edge axons kept), against the expected 1.8 +/- 0.18 under the claims file's unit;
the cKO means are 0.78 to 1.13, against 1.42 +/- 0.142. Every mouse with whole labelled images has
0.82 to 1.09 myelinated axons per um2: 0.53 of the paper's value in the Flox mouse and 0.58 to 0.69
of it in the cKO mice (up to 0.77 with edge axons kept), so the masks and the paper's counts differ
before any question of the number of mice. The masks hold 3.62 axons per um2 in all in sub-370, 26%
below the 4.86 that the paper's Fig. 2C and 2D together imply, and 4.8 to 6.9 in the cKO mice with
whole images, against 6.5. In the Flox mouse both factors are low: all axons per um2 at 0.74 of the
paper's and the fraction myelinated at 0.71 of it (26% against 37%), and together they give the 0.53.
The crops above show no obvious gaps in the masks. What remains open is the image scale (no scale bar to check it), the unit
of Fig. 2D, and which regions the paper scored. The cKO to Flox ratio is 0.81 with edge axons left
out and all four cKO mice (inside 0.79 +/- 0.079), 0.97 without sub-373C, and 1.08 or 0.97 with
edge axons kept. No reading gives a Student p below 0.63. The one-Flox-mouse density test over
images raises a scipy precision warning, as sub-370's two images have nearly equal densities; that
variant's p is not reliable.

In [ ]:
def morph_tables(seed_offset=0):
    out = {}
    for diameter in ["eq", "feret"]:
        for keep_edge in [False, True]:
            for sample in [False, True]:
                rng = np.random.default_rng(srf_tem.RANDOM_SEED + seed_offset)
                key = f"O5{'a' if diameter == 'eq' else 'b'} O4{'b' if keep_edge else 'a'} O6{'b' if sample else 'a'}"
                out[key] = srf_tem.mouse_morphometry(axons, diameter, keep_edge, sample, rng)
    return out

morph = morph_tables()
print("Per mouse values under O5a (equivalent circle), O4a (edges left out), O6a (all axons):")
print(morph["O5a O4a O6a"].assign(genotype=[genotype[s] for s in morph["O5a O4a O6a"].index]).round(4).to_string())

def morph_section(col, expected_flox, expected_cko, claim_rng_offset):
    res = {k: group_table(v[col].astype(float), col, expected_flox, expected_cko) for k, v in morph.items()}
    show = pd.concat({k: v for k, v in res.items()}, names=["reading"])
    print(show[["n_flox", "n_cko", "flox_mean", "cko_mean", "cko_minus_flox", "flox_in_tol", "cko_in_tol",
                "student_p", "welch_p"]].round(4).to_string())
    print(f"\n{len(show)} combinations of reading and subset: Flox in tolerance {int(show.flox_in_tol.sum())}, "
          f"cKO in tolerance {int(show.cko_in_tol.sum())}, cKO above Flox {int((show.cko_minus_flox > 0).sum())}; "
          f"Flox mean {show.flox_mean.min():.4g} to {show.flox_mean.max():.4g}, cKO mean {show.cko_mean.min():.4g} to "
          f"{show.cko_mean.max():.4g}, cKO minus Flox {show.cko_minus_flox.min():.4g} to {show.cko_minus_flox.max():.4g}, "
          f"Student p {show.student_p.min():.3g} to {show.student_p.max():.3g}")
    # Spread of the O6b result over 100 draws (O5a, O4a), with a generator of this claim's own.
    rng = np.random.default_rng(srf_tem.RANDOM_SEED + claim_rng_offset)
    draws = []
    for _ in range(100):
        v = srf_tem.mouse_morphometry(axons, "eq", False, True, rng)[col].astype(float)
        f = v[[genotype[s] == "WT" for s in v.index]].mean()
        c = v[[genotype[s] == "KO" for s in v.index]].mean()
        draws.append((f, c, student_one_vs_many(v[[genotype[s] == "KO" for s in v.index]], v[[genotype[s] == "WT" for s in v.index]])[2]))
    d = np.array(draws)
    print(f"\nO6b over 100 draws (O5a, O4a): Flox mean {d[:,0].min():.4f} to {d[:,0].max():.4f}; "
          f"cKO mean {d[:,1].min():.4f} to {d[:,1].max():.4f}; Student p {d[:,2].min():.3g} to {d[:,2].max():.3g}")
    return res

## fig2e-cc-g-ratio

> The average g-ratio of myelinated axons in adult corpus callosum is higher in SRF-cKO than in
> SRF-Flox mice (0.760 against 0.703).

Expected: SRF-cKO 0.760 +/- 0.076, SRF-Flox 0.703 +/- 0.0703, under readings O1, O4, O5, O6 and O7;
the paper's difference, 0.057, reported beside the result.

In [ ]:
fig2e = morph_section("g_ratio", (0.703, 0.0703), (0.760, 0.076), 1)
print("\nPaper's cKO minus Flox difference: 0.057")
print("\nVariant, images as units (O5a, O4a, all axons):")
per_img = axons[(axons.myelin_px > 0) & ~axons.fibre_edge].groupby(["subject", "sample", "genotype"]).g_eq.mean().reset_index()
print({k: round(v, 4) for k, v in image_test(per_img, "g_eq").items()})

**Result: blocked (data missing: masks for one SRF-Flox mouse).** Its mean g-ratio is 0.696 to
0.724 over the readings of O4, O5 and O6, inside 0.703 +/- 0.0703. The cKO mean is 0.675 to 0.710,
below the Flox value under every reading (cKO minus Flox -0.009 to -0.026, where the paper reports
+0.057), with Student p 0.29 to 0.77. Over 100 random draws of 100 axons the Flox mean spans 0.702
to 0.733. With images as units the Flox mean is 0.721 and the cKO mean 0.700 (Welch p 0.077). In
these masks the difference runs the other way from the paper's in every combination; with one Flox
mouse this cannot be told apart from the spread between mice.

## figs3c-cc-myelin-thickness-unchanged

> Average myelin thickness of myelinated axons in adult corpus callosum does not differ
> significantly between SRF-cKO and SRF-Flox mice (about 93 against about 105 nm, read from the
> figure), arguing against a role of SRF in myelin wrapping.

A claim of absence, judged on an unpaired two-tailed Student t test on per-mouse means at alpha
0.05 under O1, O4, O5 and O6, with Welch's test as a variant; expected SRF-Flox 105 +/- 10.5 nm,
SRF-cKO 93 +/- 9.3 nm.

In [ ]:
figs3c = morph_section("myelin_thickness_nm", (105, 10.5), (93, 9.3), 2)
print("\nVariant, images as units (O5a, O4a, all axons):")
per_img = axons[(axons.myelin_px > 0) & ~axons.fibre_edge].groupby(["subject", "sample", "genotype"]).thick_eq_nm.mean().reset_index()
print({k: round(v, 4) for k, v in image_test(per_img, "thick_eq_nm").items()})

**Result: blocked (data missing: masks for one SRF-Flox mouse).** A claim of absence is judged on the paper's test, a
Student t test on 5 against 5 per-mouse means, which cannot be run with one labelled SRF-Flox mouse.
The Flox mouse's mean myelin thickness is 96.5 to 110.0 nm over the readings (inside 105 +/- 10.5);
the cKO mean is 97.4 to 112.5 nm (cKO minus Flox -3.4 to +11.6 nm), mostly above the expected 93
+/- 9.3. The Student test of one mouse against the cKO mice gives p 0.085 to 0.99, and over images
p 0.21 (Welch 0.24).

## figs3d-cc-larger-myelinated-diameter

> Myelinated axons in adult SRF-cKO corpus callosum have a larger mean diameter than in SRF-Flox
> (about 625 against about 530 nm, read from the figure), which accounts for the higher g-ratio and
> suggests a defect in ensheathing small-caliber axons.

Expected: SRF-Flox 530 +/- 53 nm, SRF-cKO 625 +/- 62.5 nm, inner (axon) diameter, under readings
O1, O4, O5, O6 and O7.

In [ ]:
figs3d = morph_section("axon_diameter_nm", (530, 53), (625, 62.5), 3)
print("\nVariant, images as units (O5a, O4a, all axons):")
per_img = axons[(axons.myelin_px > 0) & ~axons.fibre_edge].groupby(["subject", "sample", "genotype"]).axon_d_eq_nm.mean().reset_index()
print({k: round(v, 4) for k, v in image_test(per_img, "axon_d_eq_nm").items()})
print(f"\nnotebook ran in {time.time() - started:.0f} s")

**Result: blocked (data missing: masks for one SRF-Flox mouse).** Its mean myelinated axon diameter
is 478 to 618 nm over the readings (inside 530 +/- 53 in 10 of 16 reading and subset combinations),
and the cKO mean is 446 to 569 nm, smaller than the Flox value in 15 of the 16 combinations, against
the expected 625 +/- 62.5. The one Student p below 0.05 (0.010, O5b, O4a, O6b, without sub-373C) has
cKO smaller than Flox, the reverse of the claim. With images as units, Flox 595 nm and cKO 549 nm
(Welch p 0.085). As for the g-ratio, the masks show the reverse of the paper's direction, which one
Flox mouse cannot separate from the spread between mice.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| fig2c-cc-percent-myelinated | dandiset | blocked: data missing (masks for one SRF-Flox mouse) |
| fig2d-cc-myelinated-density | dandiset | blocked: data missing (masks for one SRF-Flox mouse) |
| fig2e-cc-g-ratio | dandiset | blocked: data missing (masks for one SRF-Flox mouse) |
| figs3c-cc-myelin-thickness-unchanged | dandiset | blocked: data missing (masks for one SRF-Flox mouse) |
| figs3d-cc-larger-myelinated-diameter | dandiset | blocked: data missing (masks for one SRF-Flox mouse) |
| Optic nerve TEM (5 claims) | dandiset | blocked: data missing |
| Immunostaining, RNAscope, RT-PCR (6 claims) | dandiset | blocked: data missing |
| Culture imaging (3 claims) | dandiset | blocked: data missing |
| ChIP-seq and bulk RNA-seq (7 claims) | elsewhere (GEO GSE241561) | tested in `geo_sequencing_claims.ipynb`; see the record |
| GSEA and single-nucleus RNA-seq (4 claims) | elsewhere (GEO GSE241561) | blocked: not runnable |

**How the analysis differs from the paper's.** The paper scored axons by hand in Fiji over at least
1000 um2 and 2,500 axons per mouse, for 5 mice per genotype. Here the dataset's masks, made later for
a segmentation project, stand in for that scoring; they cover 8 full images and two small regions,
from one SRF-Flox and four SRF-cKO mice. Diameters come from object areas or minimum Feret widths,
not a line drawn by hand.

In the masks, percent myelinated is lower in every cKO mouse than in the one Flox mouse, as in the
paper, but both genotypes are below the paper's values; myelinated axons per um2 are 0.53 of the
paper's value in the Flox mouse and 0.58 to 0.69 of it in the cKO mice; and the g-ratio and myelinated axon diameter differences run the
other way from the paper's. None of this decides an outcome, as one Flox mouse cannot be compared
with five.

**What was found about the dataset.** The raw images of all 10 mice are there, more than 1000 um2
per mouse, but no per-axon measurement of the paper is deposited, and the masks cover one of the
five SRF-Flox mice. The archive summary counts 16 samples where samples.tsv lists 86 and no species;
`participants.tsv` misspells the species ("mus muculus"). Only the corpus callosum TEM of the paper
is in the dandiset; the sequencing data are in GEO.